# Proyecto Final · Módulo 4  
## Estadística y Probabilidad para Data Science orientada a Machine Learning

## Limpieza de Datos


### Extraccion y carga de DT

In [ ]:
# Importar librerías necesarias para el análisis de datos y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
# Definir rutas de los archivos CSV y cargar los datos
RUTA = {
    "order_items": "../data/raw/order_items.csv",
    "product_catalog": "../data/raw/product_catalog.csv"
    }
    
order_items = pd.read_csv(RUTA["order_items"])
product_catalog =pd.read_csv(RUTA["product_catalog"])

In [ ]:
# Unir los datos de order_items y product_catalog y reorganizar las columnas
df_limpio = pd.merge(order_items, product_catalog, on='product_id', how='left')

cols_original = df_limpio.columns.tolist()

# Nuevo orden de las columnas
nuevo_orden = [
    'order_id','product_id','product_name','product_category',
    'product_subcategory','brand','supplier'
] + [c for c in cols_original if c not in [
    'order_id','product_id','product_name','product_category',
    'product_subcategory','brand','supplier'
]]

df_limpio = df_limpio[nuevo_orden]

print(df_limpio.columns.tolist())

### Revision de datos

In [ ]:
df_limpio.info()

In [ ]:
df_limpio.isnull().sum()

In [ ]:
df_limpio.shape

In [ ]:
df_limpio.duplicated().sum()

In [ ]:
df_limpio.dtypes

### Comparación de columnas duplicadas (`_x` / `_y`)

Al unir `order_items` con `product_catalog` aparecieron columnas repetidas (`unit_price_x/_y`, `product_cost_x/_y`). Antes de decidir qué hacer con ellas, se verifica si son realmente iguales o si una depende de la otra.

In [ ]:
print("\nComparación de precios unitarios:")
print("-" * 40)
print((df_limpio["unit_price_x"] == df_limpio["unit_price_y"]).value_counts()) # True    397569

print("\nComparación de costos de producto:")
print("-" * 40)
print((df_limpio["product_cost_x"] == df_limpio["product_cost_y"]).value_counts()) # False    231956 | True     165613
                                                                                   # Conclusión: no son columnas duplicadas.

In [ ]:
print("\nComparación de costos de producto (detalle):")
print("-" * 50)
print((df_limpio["product_cost_x"] == df_limpio["product_cost_y"]))
print(
    (df_limpio["product_cost_x"] == df_limpio["product_cost_y"])
    .value_counts()
)

In [ ]:
# Comprobar si product_cost_x depende de quantity
print("Relación entre costo y cantidad:")
print("-" * 40)

print(
    (
        df_limpio["product_cost_x"]
        ==
        df_limpio["product_cost_y"] * df_limpio["quantity"]
    ).value_counts()
)
# Aparecen algunos False debido a la precisión de los números float
# True     368799
# False     28770

In [ ]:
# Repetir la comprobación usando tolerancia numérica
comparacion = np.isclose(
    df_limpio["product_cost_x"],
    df_limpio["product_cost_y"] * df_limpio["quantity"]
)

print(pd.Series(comparacion).value_counts())
# Pues sí, la relación se cumple para todas las filas considerando pequeñas diferencias de precisión numérica.
# True    397569
# Name: count, dtype: int64


In [ ]:
# Verificar una fila manualmente
num_fila = 0

print("Verificación manual de la fila")
print("=" * 75)
print(df_limpio.loc[num_fila])
print("=" * 75)

print(df_limpio.columns.tolist())
print("-" * 75)

fila = df_limpio.loc[num_fila]

print("quantity:", fila["quantity"])
print("product_cost_x:", fila["product_cost_x"])
print("product_cost_y:", fila["product_cost_y"])

print(
    "product_cost_y * quantity:",
    fila["product_cost_y"] * fila["quantity"]
)

### Filas duplicadas por combinación pedido + producto

In [ ]:
# Cuenta las filas donde se repite la combinación pedido + producto.
# No significa que sean filas completamente duplicadas.
duplicados = df_limpio.duplicated(["order_id", "product_id"]).sum()

print("Duplicados order_id + product_id:", duplicados)


filas_repetidas = df_limpio.duplicated(
    ["order_id", "product_id"],
    keep=False
).sum()

print("Filas involucradas en repeticiones:", filas_repetidas)

# Selecciona todas las filas donde se repite pedido + producto.
ejemplos = df_limpio[
    df_limpio.duplicated(["order_id", "product_id"], keep=False) # keep=False permite mostrar todas las filas de cada repetición.
].sort_values(["order_id", "product_id"])

# Muestrar los primeros 10 ejemplos.
ejemplos[
    [
        "order_id",
        "product_id",
        "product_name",
        "quantity",
        "unit_price_x",
        "discount_percentage",
        "gross_sales",
        "profit"
    ]
].head(10)


### Renombrar columnas

In [ ]:
# Renombrar columnas y eliminar columnas innecesarias
df_limpio = df_limpio.rename(columns={
    "unit_price_x": "unit_price",
    "product_cost_x": "product_cost_total",
    "product_cost_y": "product_cost",
})

df_limpio = df_limpio.drop(columns=["unit_price_y"])

print("\nColumnas después de la limpieza:")
print(df_limpio.columns.tolist())
df_limpio.head()

# df_limpio

In [ ]:
df_limpio.dtypes

Lo que pudimos encontrar en esta fase inicial de revision, es que el dataset esta bastante limpio para su manipulacion, las columnas estan con sus tipos de datos correctos, no hay valores nulos/faltantes y tampoco existen duplicados.

Cabe a destacar que la metodología del proyecto debe partir de las variables proporcionadas y documentadas por el dataset, y cualquier variable nueva que creemos debe identificarse claramente como una variable derivada.

---


In [ ]:
#Agrupacion de la columnas segun si tipo de datos para su analisis estadistico y visualizacion
col_num = df_limpio.select_dtypes(include=[np.number]).columns.tolist()
col_cat = df_limpio.select_dtypes(include=[object]).columns.tolist()

In [ ]:
print("\nAnalisis descriptivo")
print("-" * 50)
val_des=df_limpio[col_num].describe().T
val_des["skew"] = df_limpio[col_num].skew()
val_des["kurtosis"]= df_limpio[col_num].kurtosis()
val_des["cv_porc"]=(val_des["std"]/val_des["mean"]*100).round(1)

print(val_des[["mean", "std", "skew", "kurtosis","cv_porc","min","max"]])

In [ ]:
print("\nVisualizacion de distribucion de la variable descuento amount")
print("-" * 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Distribución de la variable discount_percentage", fontsize=14, fontweight='bold')

#Hist
axes[0].hist(df_limpio['discount_percentage'], bins=25, color='skyblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_limpio['discount_percentage'].mean(), color='red', linestyle="--",  label= f"Media={df_limpio['discount_percentage'].mean() :.1f}")
axes[0].axvline(df_limpio['discount_percentage'].median(), color='orange', linestyle="--", label= f"Mediana={df_limpio['discount_percentage'].median() :.1f}")
axes[0].set_title('Distribucion')
axes[0].legend(fontsize=9)


#Box
axes[1].boxplot(df_limpio['discount_percentage'], vert=True, patch_artist=True, boxprops=dict(facecolor='skyblue', alpha=0.70))
axes[1].set_title('Boxplot')
axes[1].set_ylabel('discount_percentage')

#QQ
stats.probplot(df_limpio['discount_percentage'], plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt. show()



Hist


En el histograma podemos observar que la media es superior a la mediana lo que nos confirma una asimetria positiva sesgada a la derecha. Con una agrupacion de los datos en 0 bastante fuerte con 74,000 registros y el restro de valores entre 0.05 y 0.25 con una frecuencia que va disminuyendo gradualmente.


box

En el boxplot el 50% de los datos se ubican entre el 0.05 y el 0.20 y a partir del 0.43 son valores atipicos.

QQ

El Q-Q plot muestra una desviación clara respecto a la línea de normalidad: se observan tramos planos en ambos extremos (acumulación de valores en 0 y en el tope de 0.6) y una forma de "S" en la zona central. Esto indica que la variable no sigue una distribución normal.


# 8. Planteamiento de hipótesis

## Hipótesis 1
- **Pregunta de análisis:** ¿El porcentaje de descuento aplicado tiene relación con la ganancia (`profit`) de la línea de pedido?  
- **Variables involucradas:** `discount_percentage` (continua), `profit` (continua)  
- **Tipo de variables:** Cuantitativas continuas  
- **H₀:** No existe diferencia en la ganancia promedio entre productos con descuento y sin descuento ($\mu_{\text{con\_desc}} = \mu_{\text{sin\_desc}}$).
- **H₁:** La ganancia promedio de los productos con descuento es menor que la de los productos sin descuento ($\mu_{\text{con\_desc}} < \mu_{\text{sin\_desc}}$)  
- **Prueba estadística sugerida:** Correlación de Pearson  
- **Justificación de la prueba:** Ambas variables son continuas y el objetivo es medir la fuerza y dirección de una relación lineal.

## Hipótesis 2
- **Pregunta de análisis:** ¿La ganancia promedio por línea de pedido difiere entre categorías de producto?  
- **Variables involucradas:** `product_category` (categórica), `profit` (continua)  
- **Tipo de variables:** Una categórica (15 niveles) y una cuantitativa  
- **H₀:** La ganancia promedio es igual en todas las categorías de producto  
- **H₁:** Al menos una categoría tiene una ganancia promedio distinta a las demás  
- **Prueba estadística sugerida:** ANOVA de un factor + Tukey post hoc  
- **Justificación de la prueba:** Se comparan más de dos grupos independientes sobre una variable continua.

## Hipótesis 3
- **Pregunta de análisis:** ¿La calificación del producto (`product_rating`) se relaciona con la cantidad vendida (`quantity`) por línea de pedido?  
- **Variables involucradas:** `product_rating` (continua), `quantity` (discreta)  
- **Tipo de variables:** Cuantitativas  
- **H₀:** No existe correlación entre product_rating y quantity ($\rho = 0$)  
- **H₁:** Existe correlación entre product_rating y quantity ($\rho \neq 0$)  
- **Prueba estadística sugerida:** Correlación de Spearman  
- **Justificación de la prueba:** Al ser `quantity` una variable discreta acotada, una correlación no paramétrica es más robusta.

## Hipótesis 4
- **Pregunta de análisis:** ¿El costo de envío (`shipping_cost`) promedio difiere según el proveedor (`supplier`) del producto?  
- **Variables involucradas:** `supplier` (categórica), `shipping_cost` (continua)  
- **Tipo de variables:** Categórica (múltiples grupos) y cuantitativa  
- **H₀:** El costo de envío promedio es igual entre todos los proveedores  
- **H₁:** Al menos un proveedor presenta un costo de envío promedio distinto  
- **Prueba estadística sugerida:** ANOVA de un factor + Tukey post hoc  
- **Justificación de la prueba:** Permite comparar una variable continua entre múltiples grupos categóricos.

## Hipótesis 5
- **Pregunta de análisis:** ¿La ganancia promedio difiere entre productos con precio unitario "alto" y "bajo" (usando la mediana como corte)?  
- **Variables involucradas:** `unit_price` recodificada en binaria (Alto/Bajo), `profit` (continua)  
- **Tipo de variables:** Categórica binaria y cuantitativa continua  
- **H₀:** La ganancia promedio es igual en ambos grupos de precio  
- **H₁:** La ganancia promedio difiere entre productos de precio alto y bajo  
- **Prueba estadística sugerida:** t-test para muestras independientes (Welch)  
- **Justificación de la prueba:** Se comparan exactamente dos grupos independientes sobre una variable continua.


## Hipótesis 1
**HIPOTESIS TARGET**
- **Pregunta de análisis:** ¿El porcentaje de descuento aplicado tiene relación con la ganancia (`profit`) de la línea de pedido?  
- **Variables involucradas:** `discount_percentage` (continua), `profit` (continua)  
- **Tipo de variables:** Cuantitativas continuas  
- **H₀:** No existe diferencia en la ganancia promedio entre productos con descuento y sin descuento ($\mu_{\text{con\_desc}} = \mu_{\text{sin\_desc}}$).
- **H₁:** Existe diferencia entre la ganancia media de las líneas con descuento y las sin descuento ($\mu_{con} \neq \mu_{sin}$)
- **Prueba estadística sugerida:** Prueba t de Welch
- **Justificación de la prueba:**  Se comparan las medias de una variable continua entre dos grupos independientes de tamaños y varianzas distintos, por lo que se usa la prueba t de Welch. El tamaño de muestra es grande, lo que respalda su uso aunque `profit` no sea normal.

In [90]:
print("\nPruebas de hipotesis 1")
print("-" * 50)

print("Comparacion de ganancias entre productos con y sin descuento")

def prueba_hipotesis(nombre,stat, p, alpha=0.05):
    conclusion="SE RECHAZA H0" if p < alpha else "NO SE RECHAZA H0"
    print(f"\nPrueba de Hipotesis: {nombre}")
    print(f"Estadistico de prueba: {stat:.4f}")
    print(f"Valor p: {p:.4f}")
    print(f"Conclusión: {conclusion} (alpha={alpha})")
    
g1_si=df_limpio[df_limpio['discount_percentage']>0]['profit']
g2_no=df_limpio[df_limpio['discount_percentage']==0]['profit']
t, p_t = stats.ttest_ind(g1_si, g2_no, equal_var=False) 

prueba_hipotesis("Comparacion de ganancias entre productos con y sin descuento", t, p_t)
print(f"Media de ganancias con descuento: {g1_si.mean():.2f}")
print(f"Media de ganancias sin descuento: {g2_no.mean():.2f}")
print(f"Existe diferencia {g1_si.mean() - g2_no.mean():.2f}")


Pruebas de hipotesis 1
--------------------------------------------------
Comparacion de ganancias entre productos con y sin descuento

Prueba de Hipotesis: Comparacion de ganancias entre productos con y sin descuento
Estadistico de prueba: -48.1146
Valor p: 0.0000
Conclusión: SE RECHAZA H0 (alpha=0.05)
Media de ganancias con descuento: 198.37
Media de ganancias sin descuento: 272.81
Existe diferencia -74.44


Interpretacion:

Se compararon las ganancias medias por línea de pedido entre productos con descuento y sin descuento. La prueba t welch mostró una diferencia estadísticamente significativa (t = -48.11; p < 0.001), por lo que se rechaza H₀. Las líneas con descuento tuvieron una ganancia media de 198.37, frente a 272.81 en las líneas sin descuento, es decir, 74.44 menos en promedio. Esto indica que aplicar descuento se asocia con una menor ganancia por línea de pedido

## Hipótesis 2
- **Pregunta de análisis:** ¿La ganancia promedio por línea de pedido difiere entre categorías de producto?  
- **Variables involucradas:** `product_category` (categórica), `profit` (continua)  
- **Tipo de variables:** Una categórica (15 niveles) y una cuantitativa  
- **H₀:** La ganancia promedio es igual en todas las categorías de producto  
- **H₁:** Al menos una categoría tiene una ganancia promedio distinta a las demás  
- **Prueba estadística sugerida:** ANOVA de un factor + Tukey post hoc  
- **Justificación de la prueba:** Se comparan más de dos grupos independientes sobre una variable continua.


In [96]:


print("\nPruebas de hipotesis 2")
print("-" * 50)

print("Categoria que afecta a las ganancias")

def prueba_hipotesis(nombre,stat, p, alpha=0.05):
    conclusion="SE RECHAZA H0" if p < alpha else "NO SE RECHAZA H0"
    print(f"\nPrueba de Hipotesis: {nombre}")
    print(f"Estadistico de prueba: {stat:.4f}")
    print(f"Valor p: {p:.4f}")
    print(f"Conclusión: {conclusion} (alpha={alpha})")
    

grupos_cat = [df_limpio.loc[df_limpio['product_category'] == c,'profit'].values
              for c in df_limpio['product_category'].unique()]

f, p_f = stats.f_oneway(*grupos_cat)

prueba_hipotesis("Categoria afecta las ganancias", f, p_f)

if p_f < 0.05:
    if p_f < 0.05:
        tukey = pairwise_tukeyhsd(
        endog=df_limpio['profit'],
        groups=df_limpio['product_category'],
        alpha=0.05,
    )
    print("Post hoc de Tukey (pares con diferencia significativa)")
    print(tukey.summary())
else:
    print("\n No se encontraron diferencias significativas")


Pruebas de hipotesis 2
--------------------------------------------------
Categoria que afecta a las ganancias

Prueba de Hipotesis: Categoria afecta las ganancias
Estadistico de prueba: 10073.9671
Valor p: 0.0000
Conclusión: SE RECHAZA H0 (alpha=0.05)
Post hoc de Tukey (pares con diferencia significativa)
                   Multiple Comparison of Means - Tukey HSD, FWER=0.05                   
        group1                 group2          meandiff p-adj    lower     upper   reject
-----------------------------------------------------------------------------------------
            Automotive            Baby & Kids   -46.328    0.0  -53.0831   -39.573   True
            Automotive Beauty & Personal Care -147.5615    0.0 -154.1482 -140.9749   True
            Automotive          Books & Media -185.4556    0.0 -191.9722 -178.9389   True
            Automotive            Electronics  298.7992    0.0  292.1073  305.4912   True
            Automotive                Fashion  -75.5577    0.

Interpretacion:

El ANOVA de un factor indicó diferencias significativas en la ganancia media entre categorías de producto (F = 10,073.97; p < 0.001), por lo que se rechaza H₀. La prueba post hoc de Tukey mostró que 104 de las 105 comparaciones por pares fueron significativas; la única excepción fue Automotive vs. Sports & Outdoors (diferencia = 0.62; p = 1.0). Electronics presentó la mayor ganancia media, seguida de Jewelry (87.63 menos que Electronics) y Home Appliances (144.86 menos), mientras que Books & Media, Grocery y Beauty & Personal Care presentaron las más bajas, con diferencias de 484.25, 471.96 y 446.36 respecto a Electronics.

## Hipótesis 3
- **Pregunta de análisis:** ¿La calificación del producto (`product_rating`) se relaciona con la cantidad vendida (`quantity`) por línea de pedido?  
- **Variables involucradas:** `product_rating` (continua), `quantity` (discreta)  
- **Tipo de variables:** Cuantitativas  
- **H₀:** No existe correlación entre product_rating y quantity ($\rho = 0$)  
- **H₁:** Existe correlación entre product_rating y quantity ($\rho \neq 0$)  
- **Prueba estadística sugerida:** Correlación de Spearman  
- **Justificación de la prueba:** Al ser `quantity` una variable discreta acotada, una correlación no paramétrica es más robusta.

In [93]:
print("\nPruebas de hipotesis 3")
print("-" * 50)

print("Prueba: calificación del producto y cantidad vendida")

rho, p_s = stats.spearmanr(df_limpio['product_rating'], df_limpio['quantity'])

prueba_hipotesis("Correlación de Spearman (rating vs. cantidad)", rho, p_s)
print(f"Coeficiente rho: {rho:.4f}")
print(f"n = {len(df_limpio)}")


Pruebas de hipotesis 3
--------------------------------------------------
Prueba: calificación del producto y cantidad vendida

Prueba de Hipotesis: Correlación de Spearman (rating vs. cantidad)
Estadistico de prueba: -0.0002
Valor p: 0.8905
Conclusión: NO SE RECHAZA H0 (alpha=0.05)
Coeficiente rho: -0.0002
n = 397569


Interpretacion:

La correlación de Spearman entre product_rating y quantity fue prácticamente nula (ρ = -0.0002; p = 0.8905; n = 397,569), por lo que no se rechaza H₀. No se encontró evidencia de relación entre la calificación del producto y la cantidad vendida por línea de pedido. 

## Hipótesis 4
- **Pregunta de análisis:** ¿El costo de envío (`shipping_cost`) promedio difiere según el proveedor (`supplier`) del producto?  
- **Variables involucradas:** `supplier` (categórica), `shipping_cost` (continua)  
- **Tipo de variables:** Categórica (múltiples grupos) y cuantitativa  
- **H₀:** El costo de envío promedio es igual entre todos los proveedores  
- **H₁:** Al menos un proveedor presenta un costo de envío promedio distinto  
- **Prueba estadística sugerida:** ANOVA de un factor + Tukey post hoc  
- **Justificación de la prueba:** Permite comparar una variable continua entre múltiples grupos categóricos.

In [94]:
print("\nPruebas de hipotesis 4")
print("-" * 50)
print("Costo de envio segun el proveedor")

grupos_sup = [df_limpio.loc[df_limpio['supplier'] == s, 'shipping_cost'].values for s in df_limpio['supplier'].unique()]
f, p_f = stats.f_oneway(*grupos_sup)
prueba_hipotesis("Anova (Proveedor)", f, p_f)

if p_f < 0.05:
    tukey = pairwise_tukeyhsd(df_limpio['shipping_cost'], df_limpio['supplier'], alpha=0.05)
    tabla = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    print("\nPost hoc de Tukey (pares con diferencia significativa)")
    print(tabla[tabla['reject']])
else:
    print("\nNo aplicamos Tukey: Anova no encontro diferencias")


Pruebas de hipotesis 4
--------------------------------------------------
Costo de envio segun el proveedor

Prueba de Hipotesis: Anova (Proveedor)
Estadistico de prueba: 0.6845
Valor p: 0.7238
Conclusión: NO SE RECHAZA H0 (alpha=0.05)

No aplicamos Tukey: Anova no encontro diferencias


Interpretacion:

El ANOVA de un factor no encontró diferencias significativas en el costo de envío promedio entre proveedores (F = 0.6845; p = 0.7238), por lo que no se rechaza H₀. No existe evidencia de que el proveedor del producto influya en el costo de envío. 

## Hipótesis 5
- **Pregunta de análisis:** ¿La ganancia promedio difiere entre productos con precio unitario "alto" y "bajo" (usando la mediana como corte)?  
- **Variables involucradas:** `unit_price` recodificada en binaria (Alto/Bajo), `profit` (continua)  
- **Tipo de variables:** Categórica binaria y cuantitativa continua  
- **H₀:** La ganancia promedio es igual en ambos grupos de precio  
- **H₁:** La ganancia promedio difiere entre productos de precio alto y bajo  
- **Prueba estadística sugerida:** t-test para muestras independientes (Welch)  
- **Justificación de la prueba:** Se comparan exactamente dos grupos independientes sobre una variable continua.

In [95]:
print("\nPruebas de hipotesis 5")
print("-" * 50)
print("Comparacion de ganancias entre productos de precio alto y bajo")

mediana_precio = df_limpio['unit_price'].median()
print(f"Mediana de unit_price (punto de corte): {mediana_precio:.2f}")

g1_alto = df_limpio.loc[df_limpio['unit_price'] > mediana_precio, 'profit']
g2_bajo = df_limpio.loc[df_limpio['unit_price'] <= mediana_precio, 'profit']
t, p_t = stats.ttest_ind(g1_alto, g2_bajo, equal_var=False,)

prueba_hipotesis("t de Welch: ganancias precio alto vs. precio bajo", t, p_t)
print(f"Media de ganancias precio alto: {g1_alto.mean():.2f} (n={g1_alto.count()})")
print(f"Media de ganancias precio bajo: {g2_bajo.mean():.2f} (n={g2_bajo.count()})")
print(f"Diferencia: {g1_alto.mean() - g2_bajo.mean():.2f}")


Pruebas de hipotesis 5
--------------------------------------------------
Comparacion de ganancias entre productos de precio alto y bajo
Mediana de unit_price (punto de corte): 164.97

Prueba de Hipotesis: t de Welch: ganancias precio alto vs. precio bajo
Estadistico de prueba: 349.3617
Valor p: 0.0000
Conclusión: SE RECHAZA H0 (alpha=0.05)
Media de ganancias precio alto: 337.18 (n=198761)
Media de ganancias precio bajo: 78.86 (n=198808)
Diferencia: 258.32


Interpretacion:

La prueba t de Welch mostró una diferencia significativa en la ganancia media entre productos de precio unitario alto y bajo, usando la mediana (164.97) como punto de corte (t = 349.36; p < 0.001), por lo que se rechaza H₀. Los productos de precio alto presentaron una ganancia media de 337.18, frente a 78.86 en los de precio bajo, una diferencia de 258.32. Los grupos tuvieron tamaños prácticamente iguales (n = 198,761 y n = 198,808). 


# 9. Intervalos de confianza

En esta sección calcularemos el intervalo de confianza del 95% para la ganancia promedio (`profit`) por línea de pedido del negocio, lo que nos permitirá estimar el rango real del rendimiento financiero de las transacciones.

In [ ]:

# Extracción de los datos de ganancia sin valores nulos
datos_profit = df_limpio['profit'].dropna()

# Cálculo de la media y el error estándar de la muestra
media_profit = np.mean(datos_profit)
error_std = stats.sem(datos_profit)
n = len(datos_profit)

# Cálculo del intervalo de confianza al 95% utilizando la distribución t de Student
confianza = 0.95
h = error_std * stats.t.ppf((1 + confianza) / 2.0, n - 1)

limite_inferior = media_profit - h
limite_superior = media_profit + h

print(f"Media de la Ganancia: ${media_profit:.2f}")
print(f"Intervalo de Confianza del 95%: (${limite_inferior:.2f}, ${limite_superior:.2f})")

In [ ]:
**Interpretación:**
Con un 95% de confianza, estimamos que el verdadero valor de la ganancia promedio por transacción en todo el e-commerce se encuentra entre los límites calculados. Esto nos da un rango estadísticamente seguro sobre la rentabilidad general del negocio.